# Notebook 03 — Wine Dataset

**Dataset:** 178 × 13, continuous features

**Paper parameters (Table 3):**
`c=3, c1=3, c2=2, c3=5,
l=100, G=1000, r=1, Q=12`

Results are compared against the paper's Tables 4 (RMSE), 5 (MAD), 6 (CoD).
Each percentage (30%, 40%, 50%, 60%) is run independently and saved to
`results/03_Wine_results.json`.

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from miga import MIGA
from miga.fitness import FitnessEvaluator
from miga.statistics import compute_stats, pooled_std, relative_cov, minkowski_distance
from miga.data_utils import apply_mar, apply_mnar, auto_generators, compute_metrics, EXCLUDE_COLS
from miga.paper_results import (
    TABLE3_PARAMS, BENCHMARK_Q,
    TABLE4_RMSE, TABLE5_MAD, TABLE6_COD,
    METHODS, PERCENTAGES,
)

RESULTS_DIR = os.path.join("..", "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Setup complete.")

## 1. Load Dataset

In [ ]:
from miga.data_utils import load_wine, EXCLUDE_COLS

name = "Wine"
X_true, feature_names = load_wine()
n, m = X_true.shape
exclude_cols = EXCLUDE_COLS[name]
print(f"Dataset: {name}  shape={X_true.shape}")
print(f"Features: {feature_names}")
print(f"Excluded from MAR: {[feature_names[i] for i in exclude_cols] or 'none'}")
print(f"Max features with missing (m/2): {m//2}")

## 2. Descriptive Statistics of Complete Dataset

In [ ]:
df_true = pd.DataFrame(X_true, columns=feature_names)
display(df_true.describe().round(3))

## 2b. Wine — Rank-Deficiency Diagnostic

Wine (178 × 13) is an edge case for MIGA.  MIGA's fitness function
requires estimating a 13 × 13 sample covariance matrix S_A from the
*complete* rows X_A.  For reliable estimation we need n_A ≫ m = 13.

Under MAR with 30–60% missing, X_A shrinks fast:

| Missing% | Eligible features | P(row complete) | Expected n_A | n_A / m |
|----------|-------------------|-----------------|--------------|---------|
| 30%      | floor(13/2) = 6   | 0.70^6 ≈ 0.118  | ≈ 21         | 1.6     |
| 40%      | 6                 | 0.60^6 ≈ 0.047  | ≈  8         | 0.6 ★   |
| 50%      | 6                 | 0.50^6 ≈ 0.016  | ≈  3         | 0.2 ★   |
| 60%      | 6                 | 0.40^6 ≈ 0.004  | ≈  1         | 0.1 ★   |

★ n_A < m → S_A is rank-deficient → S_A^{-1/2} is undefined.

**At 30%** n_A ≈ 21 > 13 so S_A is technically full-rank, but with
n_A / m ≈ 1.6 the smallest eigenvalues are dominated by estimation
noise.  Without regularisation, S_A^{-1/2} amplifies that noise by
orders of magnitude, producing large F_r values even for good imputations.

**At 40–60%** S_A is genuinely rank-deficient.  No imputation algorithm
that relies on a p×p sample covariance from X_A can fully compensate —
this is a hard statistical limit for MIGA on this dataset.

**Our fix**: we floor eigenvalues at 1% of the maximum (condition ≤ 100),
compared to 0.01% in the initial implementation. This bounds the noise
amplification at 30% and limits (but cannot eliminate) the impact of
rank-deficiency at 40–60%.  The paper does not document how they handle
this case; their reported results likely benefited from seeds where more
complete rows survived.

In [ ]:
# Empirical n_A diagnostic for Wine
print("Wine complete-row diagnostic")
print(f"  n={n}, m={m}, eligible features per run = {m//2}")
print()
print(f"  {'pct':>5}  {'n_A':>6}  {'n_A/m':>7}  {'status'}")
print("  " + "-"*42)
for pct in PERCENTAGES:
    X_miss_tmp = apply_mar(X_true, pct=pct, max_feat_frac=0.5, seed=pct,
                           exclude_cols=EXCLUDE_COLS[name])
    n_A = int(np.all(~np.isnan(X_miss_tmp), axis=1).sum())
    ratio = n_A / m
    status = "OK" if ratio > 3 else ("borderline" if ratio > 1 else "RANK DEFICIENT")
    print(f"  {pct:>4}%  {n_A:>6}  {ratio:>7.2f}  {status}")

## 3. MIGA Parameters (Paper Table 3)

In [ ]:
# Paper Table 3 parameters for Wine
PARAMS = dict(
    c=3, c1=3, c2=2,
    c3=5, l=100, G=1000,
    r=1, Q=BENCHMARK_Q,
    seed=2023, verbose=True,
)
print("MIGA parameters:", PARAMS)

# Reduce for quick testing — comment out for full paper reproduction
# PARAMS["l"] = 50; PARAMS["G"] = 200; PARAMS["Q"] = 3

## 4. Run MIGA for All Missing Percentages (30%, 40%, 50%, 60%)

In [ ]:
all_results = {}

for pct in PERCENTAGES:
    print(f"\n{'='*60}")
    print(f"  Missing percentage: {pct}%")
    print(f"{'='*60}")

    # Apply MAR missing mechanism (paper Section 5)
    # exclude_cols: categorical/label columns never made missing
    X_miss = apply_mar(X_true, pct=pct, max_feat_frac=0.5, seed=pct,
                       exclude_cols=EXCLUDE_COLS[name])
    missing_mask = np.isnan(X_miss)

    n_complete = int(np.all(~missing_mask, axis=1).sum())
    n_missing_vals = int(missing_mask.sum())
    print(f"  Complete rows    : {n_complete} / {n}")
    print(f"  Missing values   : {n_missing_vals} ({100*missing_mask.mean():.1f}%)")

    # Build generators from available data
    generators = auto_generators(X_miss, seed=pct)

    # Run MIGA
    miga = MIGA(**PARAMS)
    X_imputed = miga.fit_transform(X_miss, generators=generators)

    # Compute metrics on z-standardised scale
    metrics = compute_metrics(X_true, X_imputed, missing_mask)

    # Fitness decomposition
    X_A = X_miss[~np.any(missing_mask, axis=1)]
    X_C_filled = X_imputed[np.any(missing_mask, axis=1)]
    evaluator = FitnessEvaluator(X_A, r=PARAMS["r"])
    decomp = evaluator.decompose(X_C_filled)

    all_results[pct] = {
        "rmse":    metrics["rmse"],
        "mad":     metrics["mad"],
        "cod":     metrics["cod"],
        "F_r":     decomp["F_r"],
        "d_means": decomp["d_means"],
        "d_cov":   decomp["d_cov"],
        "d_skew":  decomp["d_skew"],
        "best_score": miga.best_score_,
        "history": miga.history_,
    }

    paper_rmse = TABLE4_RMSE[name][pct]["MIGA"]
    paper_mad  = TABLE5_MAD[name][pct]["MIGA"]
    paper_cod  = TABLE6_COD[name][pct]["MIGA"]

    print(f"\n  Results at {pct}% missing:")
    print(f"  {'Metric':<8} {'Ours':>10} {'Paper':>10} {'Δ':>10}")
    print(f"  {'-'*38}")
    print(f"  {'RMSE':<8} {metrics['rmse']:>10.4f} {paper_rmse:>10.4f} {metrics['rmse']-paper_rmse:>+10.4f}  (range-norm, matches paper)")
    print(f"  {'MAD':<8} {metrics['mad']:>10.4f} {paper_mad:>10.4f}  {metrics['mad']-paper_mad:>+10.4f}  (range-norm; paper formula undocumented)")
    print(f"  {'CoD':<8} {metrics['cod']:>10.4f} {paper_cod:>10.4f}  {metrics['cod']-paper_cod:>+10.4f}  (all-data SS, matches paper)")

## 5. Summary Table — Our Results vs. Paper Tables 4, 5, 6

In [ ]:
print(f"\n=== Wine: RMSE Comparison vs. Paper Table 4 ===")
print(f"  {'':<6} {' Our MIGA':>12} {'Paper MIGA':>12} {'CMIM':>10} {'ANNI':>10} {'Mean':>10}")
print("  " + "-"*62)
for pct in PERCENTAGES:
    r = all_results[pct]
    p4 = TABLE4_RMSE[name][pct]
    mean_val = p4.get('Mean') if p4.get('Mean') is not None else float('nan')
    print(f"  {pct}%   {r['rmse']:>12.4f} {p4['MIGA']:>12.4f} {p4['CMIM']:>10.4f} {p4['ANNI']:>10.4f} {mean_val:>10.4f}")

print(f"\n=== Wine: MAD Comparison vs. Paper Table 5 ===")
print(f"  {'':<6} {' Our MIGA':>12} {'Paper MIGA':>12} {'CMIM':>10} {'ANNI':>10}")
print("  " + "-"*52)
for pct in PERCENTAGES:
    r = all_results[pct]
    p5 = TABLE5_MAD[name][pct]
    print(f"  {pct}%   {r['mad']:>12.4f} {p5['MIGA']:>12.4f} {p5['CMIM']:>10.4f} {p5['ANNI']:>10.4f}")

print(f"\n=== Wine: CoD Comparison vs. Paper Table 6 ===")
print(f"  {'':<6} {' Our MIGA':>12} {'Paper MIGA':>12} {'CMIM':>10} {'ANNI':>10}")
print("  " + "-"*52)
for pct in PERCENTAGES:
    r = all_results[pct]
    p6 = TABLE6_COD[name][pct]
    print(f"  {pct}%   {r['cod']:>12.4f} {p6['MIGA']:>12.4f} {p6['CMIM']:>10.4f} {p6['ANNI']:>10.4f}")

## 6. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
pcts = PERCENTAGES

our_rmse = [all_results[p]["rmse"] for p in pcts]
pap_rmse = [TABLE4_RMSE[name][p]["MIGA"] for p in pcts]
our_mad  = [all_results[p]["mad"]  for p in pcts]
pap_mad  = [TABLE5_MAD[name][p]["MIGA"]  for p in pcts]
our_cod  = [all_results[p]["cod"]  for p in pcts]
pap_cod  = [TABLE6_COD[name][p]["MIGA"]  for p in pcts]

for ax, our, pap, ylabel, title in [
    (axes[0], our_rmse, pap_rmse, "RMSE", "RMSE"),
    (axes[1], our_mad,  pap_mad,  "MAD",  "MAD"),
    (axes[2], our_cod,  pap_cod,  "CoD",  "CoD"),
]:
    ax.plot(pcts, our, "o-", color="steelblue",  label="Our MIGA")
    ax.plot(pcts, pap, "s--", color="tomato",    label="Paper MIGA")
    ax.set_xlabel("Missing %")
    ax.set_ylabel(ylabel)
    ax.set_title(f"Wine: {title}")
    ax.set_xticks(pcts)
    ax.legend()

plt.suptitle(f"Wine Dataset — MIGA Reimplementation vs. Paper", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f"../results/03_Wine_metrics.png", dpi=120, bbox_inches="tight")
plt.show()

## 7. Convergence Across Runs

In [ ]:
fig, axes = plt.subplots(1, len(PERCENTAGES), figsize=(14, 3))
for ax, pct in zip(axes, PERCENTAGES):
    history = all_results[pct]["history"]
    ax.bar(range(1, len(history)+1), history, color="steelblue", alpha=0.8)
    ax.set_title(f"{pct}% missing")
    ax.set_xlabel("Run")
    if ax == axes[0]:
        ax.set_ylabel("Best F_r per run")
plt.suptitle(f"Wine — GA convergence per run", fontsize=12)
plt.tight_layout()
plt.savefig(f"../results/03_Wine_convergence.png", dpi=120)
plt.show()

In [ ]:
# Save all results
result_path = os.path.join(RESULTS_DIR, "03_Wine_results.json")
# Convert history lists to serialisable format
serialisable = {str(k): v for k, v in all_results.items()}
with open(result_path, "w") as f:
    json.dump(serialisable, f, indent=2)
print(f"Results saved to {result_path}")